# Train K-Means trên RFM full-period

Notebook này huấn luyện K-Means trên bảng `data/processed/rfm_uk_full.csv`, tức bảng RFM được xây dựng từ toàn bộ dữ liệu giao dịch của United Kingdom.

Mục tiêu:

- Dùng RFM full-period của UK làm dữ liệu huấn luyện.
- Log-transform cả ba biến RFM rồi chuẩn hóa trước K-Means.
- So sánh nhiều giá trị K bằng WCSS, Silhouette và kích thước cụm.
- Lưu bảng khách hàng đã phân cụm, cluster profile, model artifacts và metadata.
- Giải thích kết quả phân cụm theo góc nhìn marketing.

## 1. Load RFM và chuẩn bị output

Bước đầu tiên là đọc bảng RFM full-period và chuẩn bị thư mục lưu các artifact phục vụ dashboard ở các phase sau.

Đoạn code dưới đây import thư viện, đọc `data/processed/rfm_uk_full.csv` và tạo thư mục `models/` để lưu model mới.

In [ ]:
from pathlib import Path
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RFM_PATH = PROJECT_ROOT / "data" / "processed" / "rfm_uk_full.csv"
MODEL_DIR = PROJECT_ROOT / "models"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR.mkdir(exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 15
plt.rcParams["axes.labelsize"] = 12

rfm = pd.read_csv(RFM_PATH)
rfm.head()

Đoạn code dưới đây kiểm tra nhanh số khách hàng, missing values và khoảng giá trị RFM trước khi train.

In [ ]:
rfm_train_summary = rfm[["Recency", "Frequency", "Monetary"]].describe().T
rfm_train_summary["missing"] = rfm[["Recency", "Frequency", "Monetary"]].isna().sum()
rfm_train_summary.round(2)

**Nhận xét:** dữ liệu train mới có 3,916 khách hàng. So với RFM cũ, tập này có Recency lớn hơn, Frequency lớn hơn và Monetary lớn hơn vì đã dùng toàn bộ giai đoạn UK thay vì chỉ nửa đầu dữ liệu.

## 2. Preprocessing RFM

K-Means dùng khoảng cách Euclid, nên dữ liệu cần được đưa về không gian phù hợp trước khi train. Với RFM full-period, cả ba biến đều được log-transform bằng `log1p`, sau đó chuẩn hóa bằng `StandardScaler`.

Đoạn code dưới đây tạo ma trận feature, áp dụng `log1p` cho cả ba biến và fit scaler trên dữ liệu đã log.

In [ ]:
FEATURE_COLUMNS = ["Recency", "Frequency", "Monetary"]

X_raw = rfm[FEATURE_COLUMNS].copy()
X_log = np.log1p(X_raw)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

pd.DataFrame(X_scaled, columns=FEATURE_COLUMNS).head()

**Nhận xét:** từ bước này trở đi, WCSS, Silhouette, K-Means label và distance-to-centroid đều được tính trên cùng không gian đã `log1p` và scale. Không dùng RFM thô để đo khoảng cách.

## 3. Sweep K bằng WCSS, Silhouette và guardrail kích thước cụm

K-Means cần chọn trước số cụm K. Notebook thử K từ 2 đến 8, sau đó đánh giá bằng WCSS, Silhouette và kích thước từng cụm. Cụm quá nhỏ bị loại vì khó dùng cho marketing.

Đoạn code dưới đây train thử các giá trị K từ 2 đến 8, tính WCSS, Silhouette và kiểm tra guardrail kích thước cụm.

In [ ]:
k_results = []

for k in range(2, 9):
    candidate_model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = candidate_model.fit_predict(X_scaled)
    cluster_counts = pd.Series(labels).value_counts().sort_index()
    min_cluster_size = int(cluster_counts.min())
    max_cluster_size = int(cluster_counts.max())
    min_cluster_pct = min_cluster_size / len(rfm) * 100
    max_cluster_pct = max_cluster_size / len(rfm) * 100

    k_results.append(
        {
            "K": k,
            "WCSS": candidate_model.inertia_,
            "Silhouette": silhouette_score(X_scaled, labels),
            "MinClusterSize": min_cluster_size,
            "MaxClusterSize": max_cluster_size,
            "MinClusterPct": min_cluster_pct,
            "MaxClusterPct": max_cluster_pct,
            "PassSmallClusterGuardrail": min_cluster_size >= 30 and min_cluster_pct >= 1,
            "LargeClusterReviewFlag": max_cluster_pct > 70,
            "ClusterCounts": dict(cluster_counts),
        }
    )

k_results_df = pd.DataFrame(k_results)
k_results_df.drop(columns=["ClusterCounts"]).round(4)

**Nhận xét:** toàn bộ K từ 2 đến 8 đều qua guardrail cụm nhỏ. K=2 có Silhouette cao nhất, khoảng 0.435, trong khi K=3 đạt khoảng 0.343. Chênh lệch khoảng 0.091 lớn hơn ngưỡng 0.03 đã chốt, nên không nên ép giữ K=3 chỉ để giống model cũ.

Đoạn code dưới đây hiển thị số khách hàng trong từng cụm của mỗi giá trị K để kiểm tra mô hình có tạo cụm quá nhỏ hoặc quá lớn hay không.

In [ ]:
cluster_count_table = pd.DataFrame(
    [
        {"K": row["K"], **{f"Cluster_{label}": count for label, count in row["ClusterCounts"].items()}}
        for row in k_results
    ]
)

cluster_count_table

Đoạn code dưới đây vẽ Elbow plot bằng WCSS. WCSS luôn giảm khi K tăng, nên điểm quan trọng là xem tốc độ giảm bắt đầu chậm lại ở đâu.

In [ ]:
ax = sns.lineplot(data=k_results_df, x="K", y="WCSS", marker="o")
ax.set_title("Elbow Method: WCSS by K")
ax.set_xlabel("K")
ax.set_ylabel("WCSS")
plt.tight_layout()
plt.show()

Đoạn code dưới đây vẽ Silhouette score theo K. Silhouette càng cao thì cụm càng tách biệt tốt hơn trong không gian đã log + scale.

In [ ]:
ax = sns.lineplot(data=k_results_df, x="K", y="Silhouette", marker="o", color="#55a868")
ax.set_title("Silhouette Score by K")
ax.set_xlabel("K")
ax.set_ylabel("Silhouette Score")
plt.tight_layout()
plt.show()

## 4. Chọn K cuối cùng

Silhouette cao nhất ở K=2, nhưng K=2 chỉ tạo hai nhóm khá tổng quát: nhóm ít hoạt động và nhóm active/high-value. Với mục tiêu phân nhóm phục vụ marketing, dự án cần mức phân tầng rõ hơn để tách được nhóm inactive, nhóm tiềm năng/normal và nhóm VIP.

Vì vậy, K=3 được lựa chọn để cân bằng giữa chất lượng phân cụm và khả năng diễn giải nghiệp vụ. K=3 có silhouette thấp hơn K=2, nhưng vẫn không vi phạm điều kiện về kích thước cụm và tạo ra ba nhóm có thể chuyển thành hành động marketing rõ ràng.

Đoạn code dưới đây khóa `selected_k = 3` và lưu lại lý do chọn K=3 trong metadata của mô hình.

In [ ]:
valid_k_results = k_results_df[k_results_df["PassSmallClusterGuardrail"]].copy()
best_valid_row = valid_k_results.loc[valid_k_results["Silhouette"].idxmax()]
k3_row = valid_k_results[valid_k_results["K"].eq(3)].iloc[0]

selected_k = 3
selection_reason = (
    "K=3 is selected because it creates three actionable customer groups for marketing: "
    "inactive/at-risk, potential/normal, and VIP/high-value. K=2 has the highest "
    "silhouette score, but it is too broad for the segmentation objective. K=3 also "
    "passes the cluster-size guardrail."
)

selection_summary = pd.DataFrame(
    {
        "Metric": ["Selected K", "Best Valid K", "Best Valid Silhouette", "K=3 Silhouette", "Reason"],
        "Value": [
            selected_k,
            int(best_valid_row["K"]),
            round(float(best_valid_row["Silhouette"]), 4),
            round(float(k3_row["Silhouette"]), 4),
            selection_reason,
        ],
    }
)

selection_summary

**Nhận xét:** model mới chọn K=3 để phục vụ mục tiêu marketing segmentation. Đây là lựa chọn có chủ đích: K=2 tốt hơn về silhouette, nhưng K=3 tạo được ba nhóm hành động rõ hơn và không tạo cụm quá nhỏ.

## 5. Train model cuối và tạo cluster profile

Sau khi chọn K, train lại K-Means cuối trên toàn bộ dữ liệu đã log + scale. Kết quả label được gắn vào bảng RFM gốc để diễn giải theo giá trị thật.

Đoạn code dưới đây train model cuối, gán cluster cho từng khách hàng và tạo bảng `rfm_uk_segmented`.

In [ ]:
kmeans = KMeans(n_clusters=selected_k, random_state=RANDOM_STATE, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

rfm_uk_segmented = rfm.copy()
rfm_uk_segmented["Cluster"] = cluster_labels

rfm_uk_segmented.head()

Đoạn code dưới đây tạo profile cho từng cluster bằng RFM gốc, gồm số khách, tỷ trọng khách, doanh thu, tỷ trọng doanh thu, mean và median của ba chỉ số RFM.

In [ ]:
cluster_profile = (
    rfm_uk_segmented.groupby("Cluster")
    .agg(
        Customers=("CustomerID", "count"),
        Revenue=("Monetary", "sum"),
        RecencyMean=("Recency", "mean"),
        RecencyMedian=("Recency", "median"),
        FrequencyMean=("Frequency", "mean"),
        FrequencyMedian=("Frequency", "median"),
        MonetaryMean=("Monetary", "mean"),
        MonetaryMedian=("Monetary", "median"),
    )
    .reset_index()
)

cluster_profile["CustomerPct"] = cluster_profile["Customers"] / len(rfm_uk_segmented) * 100
cluster_profile["RevenuePct"] = cluster_profile["Revenue"] / rfm_uk_segmented["Monetary"].sum() * 100

cluster_profile.round(2)

**Nhận xét:** model mới chia khách hàng thành 2 nhóm rõ ràng. Cluster 0 có 2,401 khách hàng, chiếm 61.31% số khách nhưng chỉ tạo 15.19% doanh thu; nhóm này có Recency trung bình 134.42 ngày và Frequency trung bình 1.66 hóa đơn. Cluster 1 có 1,515 khách hàng, chiếm 38.69% số khách nhưng tạo 84.81% doanh thu; nhóm này mua gần hơn, mua thường xuyên hơn và chi tiêu cao hơn.

## 6. Distance threshold cho cảnh báo độ tin cậy

K-Means luôn gán một điểm mới vào cụm gần nhất. Vì vậy, app cần biết khi nào một input nằm quá xa vùng dữ liệu đã học. Threshold được tính theo percentile 95 của khoảng cách tới centroid trong từng cụm, trên cùng không gian đã log + scale.

Đoạn code dưới đây tính khoảng cách của từng khách hàng tới centroid của chính cụm đó, sau đó lấy percentile 95 làm ngưỡng cảnh báo riêng cho từng cluster.

In [ ]:
centers = kmeans.cluster_centers_
own_centers = centers[cluster_labels]
distances_to_own_centroid = np.linalg.norm(X_scaled - own_centers, axis=1)

rfm_uk_segmented["DistanceToCentroid"] = distances_to_own_centroid

distance_thresholds = (
    rfm_uk_segmented.groupby("Cluster")["DistanceToCentroid"]
    .quantile(0.95)
    .round(6)
    .to_dict()
)

pd.DataFrame(
    {
        "Cluster": list(distance_thresholds.keys()),
        "DistanceThresholdP95": list(distance_thresholds.values()),
    }
)

## 7. So sánh với K-Means cũ

Notebook cũ `phân tích+lưu.ipynb` train trên `rfm_uk.csv`, file này chỉ có 2,481 khách hàng và Recency tối đa 185 ngày. Model mới train trên `data/processed/rfm_uk_full.csv` với 3,916 khách hàng và Recency tối đa 374 ngày.

Đoạn code dưới đây so sánh lần phân tích hiện tại với lần phân tích trước về dữ liệu đầu vào, phạm vi RFM, cách tiền xử lý và số cụm được chọn.

In [ ]:
old_rfm = pd.read_csv("rfm_uk.csv")

model_comparison = pd.DataFrame(
    {
        "Aspect": [
            "Input RFM file",
            "Customers",
            "Max Recency",
            "Max Frequency",
            "Max Monetary",
            "Transform",
            "Selected K",
        ],
        "Old K-Means": [
            "rfm_uk.csv",
            f"{len(old_rfm):,}",
            f"{old_rfm['Recency'].max():,}",
            f"{old_rfm['Frequency'].max():,}",
            f"£{old_rfm['Monetary'].max():,.2f}",
            "log1p Frequency + Monetary, keep Recency raw",
            3,
        ],
        "New K-Means": [
            "rfm_uk_full.csv",
            f"{len(rfm):,}",
            f"{rfm['Recency'].max():,}",
            f"{rfm['Frequency'].max():,}",
            f"£{rfm['Monetary'].max():,.2f}",
            "log1p Recency + Frequency + Monetary",
            selected_k,
        ],
    }
)

model_comparison

**Nhận xét:** so với lần phân tích trước, lần này sử dụng dữ liệu RFM đầy đủ hơn, log-transform cả Recency và chọn K dựa trên cả metric lẫn khả năng ứng dụng marketing. Vì vậy, số cụm và ý nghĩa cụm có thể thay đổi so với kết quả ban đầu.

## 8. Lưu artifacts

Các artifact được lưu lại để dùng cho dashboard, kiểm tra độ tin cậy input và tài liệu hóa quy trình huấn luyện.

Đoạn code dưới đây lưu bảng đã phân cụm, cluster profile, scaler, K-Means và metadata của model mới.

In [ ]:
segmented_path = PROCESSED_DIR / "rfm_uk_segmented.csv"
profile_path = PROCESSED_DIR / "cluster_profile.csv"
scaler_path = MODEL_DIR / "scaler.pkl"
kmeans_path = MODEL_DIR / "kmeans.pkl"
metadata_path = MODEL_DIR / "model_metadata.json"

rfm_uk_segmented.to_csv(segmented_path, index=False)
cluster_profile.round(6).to_csv(profile_path, index=False)
joblib.dump(scaler, scaler_path)
joblib.dump(kmeans, kmeans_path)

metadata = {
    "country_scope": "United Kingdom",
    "source_file": str(RFM_PATH),
    "random_state": RANDOM_STATE,
    "feature_columns": FEATURE_COLUMNS,
    "transform_rule": "np.log1p applied to Recency, Frequency, Monetary, then StandardScaler",
    "distance_space": "log1p + StandardScaler transformed feature space",
    "selected_k": int(selected_k),
    "selection_reason": selection_reason,
    "train_rows": int(len(rfm)),
    "train_min": {col: float(rfm[col].min()) for col in FEATURE_COLUMNS},
    "train_max": {col: float(rfm[col].max()) for col in FEATURE_COLUMNS},
    "silhouette_scores": {str(int(row["K"])): float(row["Silhouette"]) for _, row in k_results_df.iterrows()},
    "wcss": {str(int(row["K"])): float(row["WCSS"]) for _, row in k_results_df.iterrows()},
    "cluster_counts": {str(int(k)): int(v) for k, v in pd.Series(cluster_labels).value_counts().sort_index().items()},
    "distance_threshold_p95_by_cluster": {str(int(k)): float(v) for k, v in distance_thresholds.items()},
}

metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

pd.DataFrame(
    {
        "Artifact": [segmented_path, profile_path, scaler_path, kmeans_path, metadata_path],
        "Exists": [path.exists() for path in [segmented_path, profile_path, scaler_path, kmeans_path, metadata_path]],
    }
)